# VRAM Scaling Benchmark — Extended with Efficient Attention Competitors

**Purpose:** Address advisor feedback items 5 and 8 by:
1. Adding the "middle family" of efficient/sub-quadratic attention methods (Exphormer-style sparse attention, Performer/NodeFormer-style kernelized linear attention, Specformer-style spectral attention) to memory scaling comparisons
2. Generating efficiency metrics (params, VRAM, timing) for Table 2 (HCP Connectomics, N=1000)
3. Re-running Figures 6 & 7 style comparisons with the new competitors

| Exp | What | Models |
|---|---|---|
| **1** | Isolated global branch scaling (N=200 to N=10k) | GraphFNet Spectral, Dense Attention, Exphormer-Sparse, Linear Attention (Performer/NodeFormer-adjacent), Specformer-Spectral |
| **2** | Full model VRAM at HCP scale (N=1000) | Same 5 models — for Table 2 |
| **3** | Parameter counts for all models at N=1000 | Same 5 models — for Table 2 |
| **4** | Training throughput at N=1000 (ms/batch) | Same 5 models — for Table 2 |

**Important Notes / Framing:**
- **Re-implementation Disclaimer:** Baseline global operators are complexity-matched re-implementations of their core attention mechanisms for controlled memory profiling, not exact reproductions of each method's full repository architecture.
- Exphormer: sparse attention via expander graph + virtual nodes
- Linear Attention (NodeFormer-adjacent): kernelized (FAVOR+ positive random feature) attention with linear complexity
- Specformer: self-attention in spectral domain over eigenvalues
- All models share the same local GCN branch and hyperparameters for apples-to-apples comparison

**Run on Colab with GPU runtime (T4 or better)**

In [ ]:
!pip install -q torch_geometric

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import time, gc, warnings, math
warnings.filterwarnings('ignore')

from torch_geometric.data import Data
from torch_geometric.utils import to_dense_batch, to_dense_adj

try:
    from torch_geometric.nn import GCNConv
    HAS_GCN = True
except ImportError:
    HAS_GCN = False
    print('Warning: GCNConv unavailable - dense fallback active')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {total_vram_gb:.1f} GB')
else:
    total_vram_gb = 15.0
    print('No GPU — analytical estimates used')

torch.manual_seed(42); np.random.seed(42)

HIDDEN_DIM = 128
NUM_HEADS  = 4
HEAD_DIM   = HIDDEN_DIM // NUM_HEADS
TRUNC_K    = 64
EDGE_DIM   = 3
print('Setup complete')

## 1. Global Branch Implementations

Five global mixing operators, all with the same hidden_dim interface:
- **SpectralMix** (GraphFNet) — O(N·H) activation memory
- **DenseAttention** (standard GT) — O(N²·heads) activation memory
- **ExpanderSparseAttention** (Exphormer-style) — O(N·s·heads) where s = expander degree + virtual nodes
- **KernelizedAttention** (Linear Attention, Performer/NodeFormer-style) — O(N·m·H) where m = random feature dim
- **SpectralAttention** (Specformer-style) — O(k²·H + N·k·H) where k = num eigenvalues attended

In [ ]:
# ============================================================
# 1. GraphFNet SpectralMix — our model
# ============================================================
class SpectralMixMH(nn.Module):
    """Forward intermediates: all [B,N,H] — O(N*H) incremental memory."""
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.filter_gen = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj   = nn.Linear(hidden_dim, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)

    def forward(self, x, U, mask):
        x_hat      = torch.bmm(U.transpose(1,2), x)       # [B,N,H]
        fil        = torch.sigmoid(self.filter_gen(x_hat)) # [B,N,H]
        x_filtered = fil * x_hat                            # [B,N,H]
        x_out      = torch.bmm(U, x_filtered)               # [B,N,H]
        return self.norm(self.out_proj(F.gelu(x_out * mask.unsqueeze(-1))))


# ============================================================
# 2. Standard Dense Multi-Head Self-Attention — O(N^2) baseline
# ============================================================
_attn_printed = [False]

class DenseMultiHeadSelfAttention(nn.Module):
    """Standard O(N^2) attention. Allocates [B,heads,N,N] attention tensor."""
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = hidden_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim)
        self.norm      = nn.LayerNorm(hidden_dim)

    def forward(self, x, mask):
        B, N, H = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim) \
                               .permute(2,0,3,1,4).unbind(0)
        attn = (q @ k.transpose(-2,-1)) * self.scale  # [B,heads,N,N]
        if not _attn_printed[0]:
            mb = attn.numel() * attn.element_size() / 1e6
            print(f'  [DENSE ATTN] shape={list(attn.shape)}, size={mb:.1f} MB')
            _attn_printed[0] = True
        if mask is not None:
            attn = attn.masked_fill(~mask.unsqueeze(1).unsqueeze(2), float('-inf'))
        attn = torch.softmax(attn, dim=-1)
        out  = (attn @ v).transpose(1,2).reshape(B, N, H)
        return self.norm(self.out_proj(out))


# ============================================================
# 3. Exphormer-Style Sparse Attention — O(N*s) per head
#    Uses expander graph sparsity pattern + virtual global nodes
# ============================================================
_exphormer_printed = [False]

class ExpanderSparseAttention(nn.Module):
    """Exphormer-style: sparse attention over expander edges + virtual nodes.
    Each node attends to ~log2(N) expander neighbors + v virtual nodes.
    Overall: O(N*(s+v)*head_dim) instead of O(N^2*head_dim)."""
    def __init__(self, hidden_dim, num_heads=4, num_virtual=4):
        super().__init__()
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.scale      = self.head_dim ** -0.5
        self.num_virtual = num_virtual
        self.qkv        = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.out_proj   = nn.Linear(hidden_dim, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)
        self.virtual_nodes = nn.Parameter(torch.randn(1, num_virtual, hidden_dim) * 0.02)
        self.virtual_qkv   = nn.Linear(hidden_dim, 3 * hidden_dim)

    def _make_expander_edges(self, N, device):
        s = max(4, int(math.log2(N)))
        src = torch.arange(N, device=device).repeat_interleave(s)
        dst = torch.randint(0, N, (N * s,), device=device)
        return src, dst, s

    def forward(self, x, mask):
        B, N, H = x.shape
        vn = self.virtual_nodes.expand(B, -1, -1)
        q, k, v = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim) \
                               .permute(2,0,3,1,4).unbind(0)
        vq, vk, vv = self.virtual_qkv(vn).reshape(B, self.num_virtual, 3, self.num_heads, self.head_dim) \
                                          .permute(2,0,3,1,4).unbind(0)
        attn_to_virtual = (q @ vk.transpose(-2,-1)) * self.scale
        attn_to_virtual = torch.softmax(attn_to_virtual, dim=-1)
        out_virtual = attn_to_virtual @ vv
        src, dst, s = self._make_expander_edges(N, x.device)
        out_sparse = torch.zeros_like(q)
        for b in range(B):
            q_src = q[b, :, src, :]
            k_dst = k[b, :, dst, :]
            v_dst = v[b, :, dst, :]
            edge_attn = (q_src * k_dst).sum(-1) * self.scale
            edge_attn = edge_attn.reshape(self.num_heads, N, s)
            edge_attn = torch.softmax(edge_attn, dim=-1)
            v_gathered = v_dst.reshape(self.num_heads, N, s, self.head_dim)
            out_sparse[b] = (edge_attn.unsqueeze(-1) * v_gathered).sum(2)
        out = (out_sparse + out_virtual).transpose(1,2).reshape(B, N, H)
        if not _exphormer_printed[0]:
            print(f'  [EXPHORMER] expander_degree={s}, virtual={self.num_virtual}')
            _exphormer_printed[0] = True
        return self.norm(self.out_proj(out * mask.unsqueeze(-1)))


# ============================================================
# 4. NodeFormer-Style Kernelized Attention — O(N*m)
#    Random Fourier features to approximate softmax; no N*N matrix
# ============================================================
_nodeformer_printed = [False]

class KernelizedAttention(nn.Module):
    """NodeFormer-style: FAVOR+ random features for linear attention."""
    def __init__(self, hidden_dim, num_heads=4, num_random_features=64):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = hidden_dim // num_heads
        self.m = num_random_features
        self.scale = self.head_dim ** -0.25
        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.register_buffer('W_rand', torch.randn(self.head_dim, self.m) / math.sqrt(self.m))

    def _phi(self, x):
        proj = x @ self.W_rand
        norm_sq = (x * x).sum(-1, keepdim=True) / 2
        return torch.exp(proj * self.scale - norm_sq) / math.sqrt(self.m)

    def forward(self, x, mask):
        B, N, H = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim) \
                               .permute(2,0,3,1,4).unbind(0)
        phi_q = self._phi(q)
        phi_k = self._phi(k)
        if mask is not None:
            m_expand = mask.unsqueeze(1).unsqueeze(-1)
            phi_k = phi_k * m_expand
            v = v * m_expand
        kv = torch.einsum('bhni,bhnj->bhij', phi_k, v)
        out = torch.einsum('bhni,bhij->bhnj', phi_q, kv)
        k_sum = phi_k.sum(dim=2)
        denom = torch.einsum('bhni,bhi->bhn', phi_q, k_sum).unsqueeze(-1) + 1e-6
        out = out / denom
        out = out.transpose(1,2).reshape(B, N, H)
        if not _nodeformer_printed[0]:
            print(f'  [NODEFORMER] random_features={self.m}')
            _nodeformer_printed[0] = True
        return self.norm(self.out_proj(out * mask.unsqueeze(-1)))


# ============================================================
# 5. Specformer-Style Spectral Attention — O(k^2 + N*k)
#    Attention over eigenvalues, not nodes
# ============================================================
_specformer_printed = [False]

class SpectralDomainAttention(nn.Module):
    """Specformer-style: self-attention over k spectral tokens (not N nodes)."""
    def __init__(self, hidden_dim, num_heads=4, k=64):
        super().__init__()
        self.k = k
        self.num_heads = num_heads
        self.head_dim  = hidden_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim)
        self.norm      = nn.LayerNorm(hidden_dim)

    def forward(self, x, U_trunc, mask):
        B, N, H = x.shape
        k = U_trunc.shape[-1]
        x_spec = torch.bmm(U_trunc.transpose(1,2), x)
        q, kk, v = self.qkv(x_spec).reshape(B, k, 3, self.num_heads, self.head_dim) \
                                     .permute(2,0,3,1,4).unbind(0)
        attn = (q @ kk.transpose(-2,-1)) * self.scale
        attn = torch.softmax(attn, dim=-1)
        out_spec = (attn @ v).transpose(1,2).reshape(B, k, H)
        x_out = torch.bmm(U_trunc, out_spec)
        if not _specformer_printed[0]:
            attn_mb = attn.numel() * attn.element_size() / 1e6
            print(f'  [SPECFORMER] spectral_attn [{k}x{k}], {attn_mb:.2f} MB')
            _specformer_printed[0] = True
        return self.norm(self.out_proj(F.gelu(x_out * mask.unsqueeze(-1))))


print('All 5 global branch implementations defined')

## 2. Shared Local GCN Branch & Full Model Wrappers

In [ ]:
class SparseGCNLayer(nn.Module):
    def __init__(self, d, edge_dim=3):
        super().__init__()
        self.conv = GCNConv(d, d, add_self_loops=True) if HAS_GCN else nn.Linear(d, d)
        self.edge_lin = nn.Linear(edge_dim, 1)
        self.norm = nn.LayerNorm(d)
    def forward(self, x, edge_index, edge_attr=None):
        if edge_attr is not None and HAS_GCN:
            ew = torch.sigmoid(self.edge_lin(edge_attr)).squeeze(-1)
            out = self.conv(x, edge_index, edge_weight=ew)
        elif HAS_GCN: out = self.conv(x, edge_index)
        else: out = self.conv(x)
        return self.norm(F.gelu(out))

class GatedPooling(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(d, d//2), nn.Tanh(), nn.Linear(d//2, 1))
    def forward(self, x, mask):
        s = self.gate(x).squeeze(-1).masked_fill(~mask, -1e9)
        return (x * torch.softmax(s, 1).unsqueeze(-1)).sum(1)

# --- Full model wrappers ---
class FullModel_GraphFNet(nn.Module):
    def __init__(self, in_dim=128, hidden_dim=128, num_layers=4, out_dim=1, num_heads=4, edge_dim=3):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim)
        self.layers = nn.ModuleList([nn.ModuleDict({
            'local': SparseGCNLayer(hidden_dim, edge_dim), 'global': SpectralMixMH(hidden_dim, num_heads),
            'gate': nn.Linear(hidden_dim, hidden_dim), 'norm': nn.LayerNorm(hidden_dim),
        }) for _ in range(num_layers)])
        self.pool = GatedPooling(hidden_dim); self.head = nn.Linear(hidden_dim, out_dim); self.drop = nn.Dropout(0.1)
    def forward(self, x_dense, U, mask, edge_index, batch_idx, edge_attr=None):
        x = self.proj(x_dense); idx = mask.nonzero(as_tuple=False)
        for L in self.layers:
            x_sp = L['local'](x[idx[:,0],idx[:,1]], edge_index, edge_attr=edge_attr)
            x_loc = torch.zeros_like(x); x_loc[idx[:,0],idx[:,1]] = x_sp
            x_glob = L['global'](x, U, mask); gate = torch.sigmoid(L['gate'](x))
            x = L['norm'](x + self.drop(gate*x_loc + (1-gate)*x_glob))
        return self.head(self.pool(x*mask.unsqueeze(-1), mask))

class FullModel_DenseGT(nn.Module):
    def __init__(self, in_dim=128, hidden_dim=128, num_layers=4, out_dim=1, num_heads=4, edge_dim=3):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim)
        self.layers = nn.ModuleList([nn.ModuleDict({
            'local': SparseGCNLayer(hidden_dim, edge_dim), 'global': DenseMultiHeadSelfAttention(hidden_dim, num_heads),
            'norm': nn.LayerNorm(hidden_dim),
        }) for _ in range(num_layers)])
        self.head = nn.Linear(hidden_dim, out_dim); self.drop = nn.Dropout(0.1)
    def forward(self, x_dense, mask, edge_index, batch_idx, edge_attr=None):
        x = self.proj(x_dense); idx = mask.nonzero(as_tuple=False)
        for L in self.layers:
            x_sp = L['local'](x[idx[:,0],idx[:,1]], edge_index, edge_attr=edge_attr)
            x_loc = torch.zeros_like(x); x_loc[idx[:,0],idx[:,1]] = x_sp
            x_glob = L['global'](x, mask)
            x = L['norm'](x + self.drop(0.5*x_loc + 0.5*x_glob))
        return self.head((x*mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).float())

class FullModel_Exphormer(nn.Module):
    def __init__(self, in_dim=128, hidden_dim=128, num_layers=4, out_dim=1, num_heads=4, edge_dim=3, num_virtual=4):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim)
        self.layers = nn.ModuleList([nn.ModuleDict({
            'local': SparseGCNLayer(hidden_dim, edge_dim), 'global': ExpanderSparseAttention(hidden_dim, num_heads, num_virtual),
            'norm': nn.LayerNorm(hidden_dim),
        }) for _ in range(num_layers)])
        self.head = nn.Linear(hidden_dim, out_dim); self.drop = nn.Dropout(0.1)
    def forward(self, x_dense, mask, edge_index, batch_idx, edge_attr=None):
        x = self.proj(x_dense); idx = mask.nonzero(as_tuple=False)
        for L in self.layers:
            x_sp = L['local'](x[idx[:,0],idx[:,1]], edge_index, edge_attr=edge_attr)
            x_loc = torch.zeros_like(x); x_loc[idx[:,0],idx[:,1]] = x_sp
            x_glob = L['global'](x, mask)
            x = L['norm'](x + self.drop(0.5*x_loc + 0.5*x_glob))
        return self.head((x*mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).float())

class FullModel_NodeFormer(nn.Module):
    def __init__(self, in_dim=128, hidden_dim=128, num_layers=4, out_dim=1, num_heads=4, edge_dim=3, num_random_features=64):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim)
        self.layers = nn.ModuleList([nn.ModuleDict({
            'local': SparseGCNLayer(hidden_dim, edge_dim), 'global': KernelizedAttention(hidden_dim, num_heads, num_random_features),
            'norm': nn.LayerNorm(hidden_dim),
        }) for _ in range(num_layers)])
        self.head = nn.Linear(hidden_dim, out_dim); self.drop = nn.Dropout(0.1)
    def forward(self, x_dense, mask, edge_index, batch_idx, edge_attr=None):
        x = self.proj(x_dense); idx = mask.nonzero(as_tuple=False)
        for L in self.layers:
            x_sp = L['local'](x[idx[:,0],idx[:,1]], edge_index, edge_attr=edge_attr)
            x_loc = torch.zeros_like(x); x_loc[idx[:,0],idx[:,1]] = x_sp
            x_glob = L['global'](x, mask)
            x = L['norm'](x + self.drop(0.5*x_loc + 0.5*x_glob))
        return self.head((x*mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).float())

class FullModel_Specformer(nn.Module):
    def __init__(self, in_dim=128, hidden_dim=128, num_layers=4, out_dim=1, num_heads=4, edge_dim=3, spec_k=64):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim)
        self.layers = nn.ModuleList([nn.ModuleDict({
            'local': SparseGCNLayer(hidden_dim, edge_dim), 'global': SpectralDomainAttention(hidden_dim, num_heads, spec_k),
            'norm': nn.LayerNorm(hidden_dim),
        }) for _ in range(num_layers)])
        self.head = nn.Linear(hidden_dim, out_dim); self.drop = nn.Dropout(0.1)
    def forward(self, x_dense, U_trunc, mask, edge_index, batch_idx, edge_attr=None):
        x = self.proj(x_dense); idx = mask.nonzero(as_tuple=False)
        for L in self.layers:
            x_sp = L['local'](x[idx[:,0],idx[:,1]], edge_index, edge_attr=edge_attr)
            x_loc = torch.zeros_like(x); x_loc[idx[:,0],idx[:,1]] = x_sp
            x_glob = L['global'](x, U_trunc, mask)
            x = L['norm'](x + self.drop(0.5*x_loc + 0.5*x_glob))
        return self.head((x*mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).float())

print('Full model wrappers defined')

## 3. Utilities

In [ ]:
def make_er(N, seed=42): return nx.erdos_renyi_graph(N, min(6/N, 0.3), seed=seed)

def edge_index_from_G(G, dev):
    edges = list(G.edges())
    if not edges: return torch.zeros(2, 0, dtype=torch.long, device=dev)
    s = [e[0] for e in edges]+[e[1] for e in edges]
    d = [e[1] for e in edges]+[e[0] for e in edges]
    return torch.tensor([s, d], dtype=torch.long, device=dev)

def laplacian_basis_full(G, dev):
    N = G.number_of_nodes()
    A = torch.tensor(nx.to_numpy_array(G), dtype=torch.float32, device=dev)
    A = A + torch.eye(N, device=dev)
    d = A.sum(1).pow(-0.5)
    A_norm = torch.diag(d) @ A @ torch.diag(d)
    _, U = torch.linalg.eigh(torch.eye(N, device=dev) - A_norm)
    signs = torch.sign(U[U.abs().argmax(0), torch.arange(N, device=dev)])
    signs[signs == 0] = 1.0
    return A_norm.unsqueeze(0), (U * signs.unsqueeze(0)).unsqueeze(0)

def _mem(dev):  return torch.cuda.memory_allocated(dev)/1e9 if dev.type=='cuda' else 0.0
def _peak(dev): return torch.cuda.max_memory_allocated(dev)/1e9 if dev.type=='cuda' else 0.0
def _reset(dev):
    if dev.type=='cuda': torch.cuda.reset_peak_memory_stats(dev)
def _flush(dev):
    gc.collect()
    if dev.type=='cuda': torch.cuda.empty_cache(); torch.cuda.synchronize()

def measure_incremental(fn, inputs, dev, warmup=2):
    _flush(dev)
    try:
        with torch.no_grad():
            for _ in range(warmup): fn(*inputs)
        _flush(dev); baseline = _mem(dev); _reset(dev)
        with torch.no_grad(): fn(*inputs)
        if dev.type=='cuda': torch.cuda.synchronize()
        return max(_peak(dev) - baseline, 0.0)
    except RuntimeError as e:
        if 'out of memory' in str(e).lower(): _flush(dev); return None
        raise

def count_params(model): return sum(p.numel() for p in model.parameters())
def count_params_str(model):
    n = count_params(model)
    return f'{n/1e6:.2f}M' if n >= 1e6 else f'{n/1e3:.1f}k'

# GPU warmup
if device.type == 'cuda':
    print('GPU warmup...')
    for _ in range(5): _ = torch.randn(64,512,device=device) @ torch.randn(512,512,device=device)
    for Nw in [50,100,200]:
        m = torch.randn(Nw,Nw,device=device); m = m+m.T
        for _ in range(3): torch.linalg.eigh(m)
    if HAS_GCN:
        c = GCNConv(HIDDEN_DIM,HIDDEN_DIM).to(device)
        for _ in range(5): c(torch.randn(100,HIDDEN_DIM,device=device), torch.randint(0,100,(2,300),device=device))
        del c
    _flush(device); print('GPU warmup complete.')
print('Utilities ready')

---
## Experiment 1: Isolated Global Branch Scaling (N=200 to N=10,000)

In [ ]:
EXP1_N = [200, 500, 1000, 2000, 5000, 10000]
exp1 = {k: [] for k in ['N', 'spectral', 'dense_attn', 'exphormer', 'nodeformer', 'specformer']}

print('='*90)
print('EXP 1: Isolated Global Branch Scaling')
print('='*90)
print(f'{"N":>8} | {"GraphFNet":>12} | {"DenseAttn":>12} | {"Exphormer":>12} | {"NodeFormer":>12} | {"Specformer":>12}')
print('-'*85)

for N in EXP1_N:
    exp1['N'].append(N)
    G = make_er(N); xd = torch.randn(1,N,HIDDEN_DIM,device=device)
    m = torch.ones(1,N,dtype=torch.bool,device=device)
    results = {}
    
    # GraphFNet
    try:
        _, U = laplacian_basis_full(G, device)
        spec = SpectralMixMH(HIDDEN_DIM, NUM_HEADS).to(device)
        results['spectral'] = measure_incremental(spec, (xd, U, m), device)
        del spec, U; _flush(device)
    except: results['spectral'] = None; _flush(device)
    
    # Dense Attn
    _attn_printed[0] = False
    try:
        attn = DenseMultiHeadSelfAttention(HIDDEN_DIM, NUM_HEADS).to(device)
        results['dense_attn'] = measure_incremental(attn, (xd, m), device)
        del attn; _flush(device)
    except: results['dense_attn'] = None; _flush(device)
    
    # Exphormer
    _exphormer_printed[0] = False
    try:
        exph = ExpanderSparseAttention(HIDDEN_DIM, NUM_HEADS, 4).to(device)
        results['exphormer'] = measure_incremental(exph, (xd, m), device)
        del exph; _flush(device)
    except: results['exphormer'] = None; _flush(device)
    
    # NodeFormer
    _nodeformer_printed[0] = False
    try:
        nf = KernelizedAttention(HIDDEN_DIM, NUM_HEADS, 64).to(device)
        results['nodeformer'] = measure_incremental(nf, (xd, m), device)
        del nf; _flush(device)
    except: results['nodeformer'] = None; _flush(device)
    
    # Specformer
    _specformer_printed[0] = False
    try:
        _, Uf = laplacian_basis_full(G, device); Ut = Uf[:,:,:TRUNC_K]
        sf = SpectralDomainAttention(HIDDEN_DIM, NUM_HEADS, TRUNC_K).to(device)
        results['specformer'] = measure_incremental(sf, (xd, Ut, m), device)
        del sf, Uf, Ut; _flush(device)
    except: results['specformer'] = None; _flush(device)
    
    for k in ['spectral','dense_attn','exphormer','nodeformer','specformer']: exp1[k].append(results.get(k))
    fmt = lambda v: f'{v*1000:.1f} MB' if v is not None else 'OOM'
    print(f'{N:>8} | {fmt(results["spectral"]):>12} | {fmt(results["dense_attn"]):>12} | '
          f'{fmt(results["exphormer"]):>12} | {fmt(results["nodeformer"]):>12} | {fmt(results["specformer"]):>12}')
    del xd, m; _flush(device)

print('\nExperiment 1 complete.')

---
## Experiment 2: Full Model VRAM Scaling (N=200 to N=5000)

In [ ]:
EXP2_N = [200, 500, 1000, 2000, 5000]
MK = dict(in_dim=HIDDEN_DIM, hidden_dim=HIDDEN_DIM, num_layers=4, out_dim=1, num_heads=NUM_HEADS, edge_dim=EDGE_DIM)
exp2 = {name: {'N':[],'vram':[]} for name in ['GraphFNet','DenseGT','Exphormer','NodeFormer','Specformer']}

print('='*90)
print('EXP 2: Full 4-Layer Models — Peak VRAM')
print('='*90)
print(f'{"N":>8} | {"GraphFNet":>12} | {"DenseGT":>12} | {"Exphormer":>12} | {"NodeFormer":>12} | {"Specformer":>12}')
print('-'*85)

for N in EXP2_N:
    G = make_er(N); ei = edge_index_from_G(G, device)
    ea = torch.randn(ei.size(1), EDGE_DIM, device=device)
    bi = torch.zeros(N, dtype=torch.long, device=device)
    xd = torch.randn(1,N,HIDDEN_DIM,device=device)
    m = torch.ones(1,N,dtype=torch.bool,device=device)
    results = {}
    
    try: _, U = laplacian_basis_full(G, device); Ut = U[:,:,:TRUNC_K]
    except:
        print(f'{N:>8} | All OOM (eigenbasis)');
        for name in exp2: exp2[name]['N'].append(N); exp2[name]['vram'].append(None)
        _flush(device); continue
    
    for name, make_fn, inp_fn in [
        ('GraphFNet',  lambda: FullModel_GraphFNet(**MK).to(device), lambda mdl: (xd,U,m,ei,bi,ea)),
        ('DenseGT',    lambda: FullModel_DenseGT(**MK).to(device),   lambda mdl: (xd,m,ei,bi,ea)),
        ('Exphormer',  lambda: FullModel_Exphormer(**MK,num_virtual=4).to(device), lambda mdl: (xd,m,ei,bi,ea)),
        ('NodeFormer', lambda: FullModel_NodeFormer(**MK,num_random_features=64).to(device), lambda mdl: (xd,m,ei,bi,ea)),
        ('Specformer', lambda: FullModel_Specformer(**MK,spec_k=TRUNC_K).to(device), lambda mdl: (xd,Ut,m,ei,bi,ea)),
    ]:
        _attn_printed[0]=_exphormer_printed[0]=_nodeformer_printed[0]=_specformer_printed[0]=False
        try:
            mdl = make_fn(); v = measure_incremental(mdl, inp_fn(mdl), device)
            results[name] = v; del mdl; _flush(device)
        except: results[name] = None; _flush(device)
        exp2[name]['N'].append(N); exp2[name]['vram'].append(results.get(name))
    
    fmt = lambda v: f'{v:.4f} GB' if v is not None else 'OOM'
    print(f'{N:>8} | {fmt(results["GraphFNet"]):>12} | {fmt(results["DenseGT"]):>12} | '
          f'{fmt(results["Exphormer"]):>12} | {fmt(results["NodeFormer"]):>12} | {fmt(results["Specformer"]):>12}')
    del U, Ut, xd, m, ei, ea; _flush(device)

print('\nExperiment 2 complete.')

---
## Experiment 3: Parameter Counts & Timing at N=1000

In [ ]:
print('='*60)
print('EXP 3: Parameter Counts (HCP config, hidden=128, 4 layers)')
print('='*60)
models_for_params = {
    'GraphFNet':  FullModel_GraphFNet(**MK),
    'DenseGT':    FullModel_DenseGT(**MK),
    'Exphormer':  FullModel_Exphormer(**MK, num_virtual=4),
    'NodeFormer': FullModel_NodeFormer(**MK, num_random_features=64),
    'Specformer': FullModel_Specformer(**MK, spec_k=TRUNC_K),
}
param_table = {}
print(f'{"Model":>15} | {"Params":>12}')
print('-'*30)
for name, mdl in models_for_params.items():
    param_table[name] = count_params(mdl)
    print(f'{name:>15} | {count_params_str(mdl):>12}')
del models_for_params; _flush(device)

# Timing
N_HCP = 1000; RUNS = 20; WARM = 5
print(f'\n{"="*60}')
print(f'EXP 3b: Forward-Pass Timing at N={N_HCP} ({RUNS} runs, median)')
print(f'{"="*60}')
G = make_er(N_HCP); ei = edge_index_from_G(G, device)
ea = torch.randn(ei.size(1), EDGE_DIM, device=device)
bi = torch.zeros(N_HCP, dtype=torch.long, device=device)
xd = torch.randn(1,N_HCP,HIDDEN_DIM,device=device)
m = torch.ones(1,N_HCP,dtype=torch.bool,device=device)
_, U = laplacian_basis_full(G, device); Ut = U[:,:,:TRUNC_K]

timing_results = {}
print(f'{"Model":>15} | {"Median ms":>10} | {"Mean ms":>10}')
print('-'*40)

for name, make_fn, inp_fn in [
    ('GraphFNet',  lambda: FullModel_GraphFNet(**MK).to(device), lambda md: (xd,U,m,ei,bi,ea)),
    ('DenseGT',    lambda: FullModel_DenseGT(**MK).to(device),   lambda md: (xd,m,ei,bi,ea)),
    ('Exphormer',  lambda: FullModel_Exphormer(**MK,num_virtual=4).to(device), lambda md: (xd,m,ei,bi,ea)),
    ('NodeFormer', lambda: FullModel_NodeFormer(**MK,num_random_features=64).to(device), lambda md: (xd,m,ei,bi,ea)),
    ('Specformer', lambda: FullModel_Specformer(**MK,spec_k=TRUNC_K).to(device), lambda md: (xd,Ut,m,ei,bi,ea)),
]:
    _attn_printed[0]=_exphormer_printed[0]=_nodeformer_printed[0]=_specformer_printed[0]=False
    try:
        mdl = make_fn(); inputs = inp_fn(mdl)
        with torch.no_grad():
            for _ in range(WARM): mdl(*inputs)
        if device.type=='cuda': torch.cuda.synchronize()
        times = []
        for _ in range(RUNS):
            if device.type=='cuda': torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad(): mdl(*inputs)
            if device.type=='cuda': torch.cuda.synchronize()
            times.append((time.perf_counter()-t0)*1000)
        med = float(np.median(times)); avg = float(np.mean(times))
        timing_results[name] = med
        print(f'{name:>15} | {med:>10.2f} | {avg:>10.2f}')
        del mdl; _flush(device)
    except Exception as e:
        print(f'{name:>15} | ERROR: {e}'); timing_results[name] = None; _flush(device)

del U, Ut, xd, m, ei, ea; _flush(device)
print('\nExperiment 3 complete.')

---
## Summary & Plots

In [ ]:
# --- Summary Table ---
print('\n' + '='*70)
print('SUMMARY: Efficiency at N=1000 (for Table 2 / Paper)')
print('='*70)
print(f'{"Model":>15} | {"Params":>10} | {"VRAM (GB)":>12} | {"Fwd ms":>10}')
print('-'*55)
for name in ['GraphFNet','DenseGT','Exphormer','NodeFormer','Specformer']:
    p = param_table.get(name, 0); ps = f'{p/1e3:.0f}k' if p < 1e6 else f'{p/1e6:.2f}M'
    vram = None
    for i,n in enumerate(exp2[name]['N']):
        if n == 1000: vram = exp2[name]['vram'][i]; break
    vs = f'{vram:.4f}' if vram else 'OOM'
    ts = f'{timing_results[name]:.2f}' if timing_results.get(name) else '—'
    print(f'{name:>15} | {ps:>10} | {vs:>12} | {ts:>10}')

In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.spines.top':False,'axes.spines.right':False,
                     'figure.dpi':130,'axes.facecolor':'white','figure.facecolor':'white'})
COLORS = {'GraphFNet':'#1f77b4','DenseGT':'#d62728','Exphormer':'#2ca02c','NodeFormer':'#ff7f0e','Specformer':'#9467bd'}
MARKERS = {'GraphFNet':'s','DenseGT':'o','Exphormer':'^','NodeFormer':'D','Specformer':'v'}

# Plot 1: Isolated global branch
fig, ax = plt.subplots(figsize=(9,6))
nmap = {
    'spectral': ('GraphFNet (Spectral)', 'GraphFNet'),
    'dense_attn': ('Dense Attention (O(N²))', 'DenseGT'),
    'exphormer': ('Exphormer (Sparse)', 'Exphormer'),
    'nodeformer': ('Linear Attention (Performer/NodeFormer-style)', 'NodeFormer'),
    'specformer': ('Specformer (Spectral Attn)', 'Specformer')
}
for key,(label,ckey) in nmap.items():
    valid = [(n,v*1000) for n,v in zip(exp1['N'],exp1[key]) if v is not None]
    if valid:
        ns,vs = zip(*valid)
        ax.plot(ns,vs,'-'+MARKERS[ckey],color=COLORS[ckey],label=label,markersize=7,linewidth=2)
ax.set_xlabel('Nodes (N)'); ax.set_ylabel('VRAM (MB)'); ax.set_xscale('log'); ax.set_yscale('log')
ax.legend(loc='upper left',framealpha=0.9); ax.grid(True,alpha=0.3)
ax.set_title('Isolated Global Branch — Memory Scaling (5 Models)')
plt.tight_layout(); plt.savefig('exp1_5models.png',dpi=200,bbox_inches='tight'); plt.show()

# Plot 2: Full model
fig, ax = plt.subplots(figsize=(9,6))
display_names = {
    'GraphFNet': 'GraphFNet (Ours)',
    'DenseGT': 'Dense Attention GT',
    'Exphormer': 'Exphormer (Sparse)',
    'NodeFormer': 'Linear Attention (Performer/NodeFormer-style)',
    'Specformer': 'Specformer (Spectral Attn)'
}
for name in ['GraphFNet','DenseGT','Exphormer','NodeFormer','Specformer']:
    valid = [(n,v) for n,v in zip(exp2[name]['N'],exp2[name]['vram']) if v is not None]
    if valid:
        ns,vs = zip(*valid)
        ax.plot(ns,vs,'-'+MARKERS[name],color=COLORS[name],label=display_names.get(name, name),markersize=7,linewidth=2)
ax.set_xlabel('Nodes (N)'); ax.set_ylabel('Peak VRAM (GB)'); ax.set_xscale('log'); ax.set_yscale('log')
ax.axvline(x=1000,color='gray',linestyle='--',alpha=0.5)
ax.annotate('HCP (N=1000)',xy=(1000,ax.get_ylim()[0]),fontsize=9,ha='center',color='gray')
ax.legend(loc='upper left',framealpha=0.9); ax.grid(True,alpha=0.3)
ax.set_title('Full 4-Layer Model — Peak VRAM (Sparse GCN shared)')
plt.tight_layout(); plt.savefig('exp2_5models.png',dpi=200,bbox_inches='tight'); plt.show()

print('Plots saved.')

In [ ]:
# LaTeX table output
display_names_latex = {
    'GraphFNet': r'\textbf{GraphFNet (Ours)}',
    'DenseGT': 'Dense Attention GT',
    'Exphormer': 'Exphormer (Expander-sparse)',
    'NodeFormer': 'Linear Attention (Performer/NodeFormer-adjacent)',
    'Specformer': 'Specformer (Spectral-domain)'
}

print(r'\begin{table}[t]')
print(r'\caption{Efficiency comparison of global mixing operators at HCP scale ($N=1{,}000$, 4-layer models, sparse GCN local branch, $d=128$). Baseline global operators are complexity-matched re-implementations of their core attention mechanisms for controlled memory profiling, not exact reproductions of each method\'s full repository architecture. GraphFNet and Specformer utilize precomputed Laplacian eigenbases.}')
print(r'\label{tab:efficiency_hcp}')
print(r'\centering')
print(r'\begin{tabular}{lccc}')
print(r'\toprule')
print(r'\textbf{Model} & \textbf{Params} & \textbf{Peak VRAM (GB)} & \textbf{Fwd (ms)} \\')
print(r'\midrule')
for name in ['GraphFNet','DenseGT','Exphormer','NodeFormer','Specformer']:
    p = param_table.get(name,0); ps = f'{p/1e3:.0f}k' if p<1e6 else f'{p/1e6:.2f}M'
    vram = None
    for i,n in enumerate(exp2[name]['N']):
        if n==1000: vram=exp2[name]['vram'][i]; break
    vs = f'{vram:.3f}' if vram else '---'
    ts = f'{timing_results[name]:.1f}' if timing_results.get(name) else '---'
    dn = display_names_latex.get(name, name)
    if name=='GraphFNet':
        print(f'{dn} & \\textbf{{{ps}}} & \\textbf{{{vs}}} & \\textbf{{{ts}}} \\\\')
    else:
        print(f'{dn} & {ps} & {vs} & {ts} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')
print('\n% NOTE: Baseline models are complexity-matched re-implementations for controlled memory profiling.')